# Day 18 - OPTIMIZE VACUUM and Small Files

**Topic:** OPTIMIZE VACUUM and Small Files  
**Main environment:** Microsoft Fabric Notebook + Fabric Lakehouse  
**Focus:** เข้าใจ small files problem และฝึก maintenance mindset ด้วย `OPTIMIZE`, `VACUUM`, retention และ Delta table history

วันนี้เราจะเรียนว่า Lakehouse table ที่ถูกเขียนบ่อย ๆ อาจสะสม small files ได้อย่างไร ทำไม small files ส่งผลต่อ performance และ maintenance cost และควรใช้ `OPTIMIZE` กับ `VACUUM` แบบระวัง retention / time travel อย่างไรใน Microsoft Fabric Notebook

## Goal of the day

1. อธิบายได้ว่า **small files problem** คืออะไร และทำไมกระทบ Lakehouse table
2. สร้าง Delta table ตัวอย่างที่เกิดหลาย small writes ได้
3. ใช้ `DESCRIBE DETAIL` และ `DESCRIBE HISTORY` เพื่อตรวจ table/file metadata เบื้องต้นได้
4. ใช้ `OPTIMIZE` เพื่อ compact small files ใน lab table ได้
5. ใช้ `VACUUM ... DRY RUN` เพื่อ preview cleanup โดยไม่ลบไฟล์จริง
6. เข้าใจว่า retention ของ `VACUUM` กระทบ time travel / recovery window อย่างไร

## Why it matters in production

ใน production Lakehouse pipeline ข้อมูลมักถูกเขียนแบบ incremental, micro-batch, append, merge หรือ update บ่อย ๆ ถ้าแต่ละ run เขียนไฟล์เล็กจำนวนมาก table จะเริ่มมี metadata/file-operation overhead มากขึ้น และ query อาจช้าลงจากการต้อง list / scan ไฟล์จำนวนมาก

เรื่องนี้สำคัญเพราะ:

- small files ทำให้ Spark ต้องจัดการ file metadata และ tasks จำนวนมากเกินจำเป็น
- compaction ช่วยรวมไฟล์เล็กให้เป็นไฟล์ที่เหมาะกับการอ่านมากขึ้น
- `OPTIMIZE` ช่วยปรับ data layout แต่ไม่ได้ลบไฟล์เก่าออกทันที
- `VACUUM` ช่วยลบ stale/unreferenced files แต่ต้องระวัง retention
- retention ที่สั้นเกินไปอาจทำให้ time travel หรือ recovery ย้อนหลังไม่ได้
- maintenance ควรเป็นส่วนหนึ่งของ operational routine ไม่ใช่ทำแบบสุ่มเมื่อ table เริ่มช้า

Production takeaway:

> `OPTIMIZE` improves table layout. `VACUUM` cleans obsolete files. ทั้งสองอย่างช่วยคนละปัญหา และต้องใช้โดยคิดถึง retention, recovery และงานอื่นที่อาจกำลังอ่านหรือเขียน table เดียวกันอยู่เสมอ

## Key concepts

| Concept | Meaning |
|---|---|
| Small files problem | ปัญหาที่ table มีไฟล์เล็กจำนวนมาก ทำให้ metadata overhead และ read overhead สูงขึ้น |
| File compaction | การรวมไฟล์เล็กหลายไฟล์ให้เป็นไฟล์ขนาดเหมาะสมขึ้น |
| `OPTIMIZE` | Delta maintenance command ที่ rewrite small files เป็นไฟล์ที่ใหญ่และเหมาะกับการอ่านมากขึ้น |
| `VACUUM` | Delta maintenance command ที่ลบ data files ที่ไม่ถูก reference แล้วและเก่ากว่า retention threshold |
| Retention threshold | ระยะเวลาที่ต้องเก็บไฟล์เก่าก่อนอนุญาตให้ `VACUUM` ลบ |
| Time travel window | ช่วงเวลาที่เรายัง query table version เก่าได้ ถ้าไฟล์ที่ version นั้นต้องใช้ยังไม่ถูก vacuum |
| `DRY RUN` | โหมด preview ว่า `VACUUM` จะลบไฟล์อะไร โดยยังไม่ลบจริง |
| `DESCRIBE DETAIL` | ใช้ดู metadata ของ Delta table เช่น location, file count, size |
| `DESCRIBE HISTORY` | ใช้ดู commit history และ operation metrics ของ Delta table |
| V-Order | Fabric file-level optimization ที่ช่วยปรับ file layout ให้เหมาะกับการอ่านข้อมูลมากขึ้น มักใช้กับ read-heavy workload |


## Concept explanation

### Small files problem คืออะไร?

Delta table ด้านหลังประกอบด้วย Parquet files และ `_delta_log` ที่เก็บ transaction log ถ้า pipeline append ข้อมูลทีละน้อยบ่อย ๆ เช่น ทุก 5 นาทีหรือทุก batch เล็ก ๆ table อาจมีไฟล์เล็กจำนวนมาก

ผลที่ตามมา:

- Spark ต้อง list files มากขึ้น
- Spark ต้องตรวจ metadata ของไฟล์มากขึ้นก่อนเริ่มอ่านข้อมูลจริง
- Spark มี overhead จากการวางแผนและจัดการ tasks มากขึ้น
- file skipping / partition pruning ช่วยได้บางส่วน แต่ยังมี overhead จากจำนวนไฟล์ที่มากเกินไป
- table maintenance ยากขึ้นในระยะยาว เช่น OPTIMIZE/VACUUM ใช้เวลามากขึ้น หรือดูแล retention ยากขึ้น

พูดง่าย ๆ:

> ไฟล์เล็กไม่ผิดเสมอ แต่ไฟล์เล็กจำนวนมากเกินไปมักทำให้ Lakehouse table อ่านและดูแลยากขึ้น

### `OPTIMIZE` ทำอะไร?

`OPTIMIZE` คือ compaction operation ของ Delta table

มันจะ:

1. หาไฟล์ที่เล็กหรือ fragmented
2. rewrite data เป็นไฟล์ใหม่ที่เหมาะสมขึ้น
3. commit version ใหม่ใน Delta log
4. ทำให้ไฟล์เก่าที่ถูกแทนที่กลายเป็น unreferenced files

สิ่งสำคัญคือ `OPTIMIZE` ไม่ได้ลบไฟล์เก่าทันที ไฟล์เก่าจะยังอยู่เพื่อรองรับ retention, time travel และ concurrent readers จนกว่าจะเข้าเงื่อนไขของ `VACUUM`

### `VACUUM` ทำอะไร?

`VACUUM` ใช้ลบไฟล์ที่:

1. ไม่ถูก reference โดย active Delta table state แล้ว
2. เก่ากว่า retention threshold

ดังนั้น `VACUUM` ไม่ใช่คำสั่งเร่ง query โดยตรง แต่ช่วย reclaim storage และลด stale files หลังจากมี operation เช่น overwrite, delete, merge หรือ optimize

### Retention สำคัญอย่างไร?

Retention คือ safety window ถ้า run `VACUUM` aggressive เกินไป เช่น retention สั้นมาก อาจทำให้:

- query version เก่าไม่ได้
- recovery จาก bad load ทำได้ยากขึ้น
- long-running reader/writer ได้รับผลกระทบ

สำหรับ learning lab วันนี้ เราจะใช้ `VACUUM ... DRY RUN` เป็นหลัก เพื่อฝึก mindset โดยไม่ลบไฟล์จริง

### OPTIMIZE vs VACUUM

| Question | `OPTIMIZE` | `VACUUM` |
|---|---|---|
| เป้าหมายหลัก | Improve file layout / read efficiency | Remove obsolete files / reclaim storage |
| ลบไฟล์จริงไหม | ไม่ | ใช่ ถ้าเข้า retention condition |
| กระทบ time travel ไหม | ไม่กระทบ | กระทบ ถ้าลบไฟล์ที่ version เก่าต้องใช้ |
| ควรใช้เมื่อไร | มี small files หรือ fragmented table | มี stale files ที่เกิน retention แล้ว |


## Mermaid diagram: Small Files Maintenance Flow

```mermaid
flowchart LR
    A[Many small append writes] --> B[Delta table with many small Parquet files]
    B --> C[Read query has more file metadata overhead]
    B --> D[OPTIMIZE]
    D --> E[Fewer right-sized active files]
    D --> F[Old files become unreferenced]
    F --> G{Older than retention?}
    G -- No --> H[Keep files for time travel / safety]
    G -- Yes --> I[VACUUM can remove stale files]
    I --> J[Storage cleanup]
```

Key idea:

> `OPTIMIZE` เปลี่ยน physical layout ของ active table ส่วน `VACUUM` ลบไฟล์เก่าที่ Delta table ไม่ต้องใช้แล้วหลังผ่าน retention window

## Setup / imports

In [1]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

StatementMeta(, 20797979-2d68-424b-8344-32f989291512, 3, Finished, Available, Finished, False)

## Create mock data

Notebook นี้จะสร้าง lab table ชื่อ `day18_sales_events_small_files` แล้วจำลองหลาย small append writes

In [2]:
table_name = "day18_sales_events_small_files"

spark.sql(f"DROP TABLE IF EXISTS {table_name}")

sales_event_schema = T.StructType([
    T.StructField("event_id", T.StringType(), False),
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("region", T.StringType(), True),
    T.StructField("amount", T.DoubleType(), True),
    T.StructField("event_date", T.StringType(), True),
    T.StructField("source_batch_id", T.StringType(), True),
])

def create_sales_batch(batch_number: int):
    # Create a small source batch for the small-files lab.
    regions = ["Bangkok", "Chiang Mai", "Phuket", "Rayong"]
    rows = []

    for i in range(1, 5):
        rows.append((
            f"B{batch_number:02d}_E{i:03d}",
            1000 + batch_number * 10 + i,
            regions[(batch_number + i) % len(regions)],
            float(100 * batch_number + 25 * i),
            f"2026-05-{batch_number:02d}",
            f"batch_{batch_number:02d}",
        ))

    return spark.createDataFrame(rows, sales_event_schema)

# Tip:
# - :02d formats an integer as at least 2 digits with leading zero, e.g. 1 -> 01.
# - % (modulo) returns the remainder, e.g. 5 % 2 = 1.
# - / (division) returns the quotient as a float, e.g. 5 / 2 = 2.5.

StatementMeta(, 20797979-2d68-424b-8344-32f989291512, 4, Finished, Available, Finished, False)

## PySpark code examples

ใน section นี้เราจะทำครบ flow แบบ learning lab:

1. เขียนหลาย small batches ลง Delta table
2. ตรวจ table detail / history
3. run query เพื่อเช็ก behavior
4. run `OPTIMIZE`
5. ตรวจ metrics หลัง optimize
6. run `VACUUM ... DRY RUN` แบบปลอดภัย

### Example 1: Generate multiple small writes

เราจะ append 6 small batches และใช้ `repartition(4)` เพื่อเพิ่มจำนวน partitions ก่อน write ซึ่งโดยทั่วไปจะทำให้แต่ละ batch สร้างหลาย output files

In [3]:
for batch_number in range(1, 7):
    df_batch = (
        create_sales_batch(batch_number)
        .withColumn("event_date", F.to_date("event_date"))
        .withColumn("ingestion_timestamp", F.current_timestamp())
    )

    (
        df_batch
        .repartition(4)  # Reshuffle rows into 4 partitions before write to simulate small files.
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(table_name)
    )

print(f"Created lab table: {table_name}")
spark.read.table(table_name).orderBy("source_batch_id", "event_id").show(30, truncate=False)
print("Row count:", spark.read.table(table_name).count())

StatementMeta(, 20797979-2d68-424b-8344-32f989291512, 5, Finished, Available, Finished, False)

Created lab table: day18_sales_events_small_files
+--------+-----------+----------+------+----------+---------------+--------------------------+
|event_id|customer_id|region    |amount|event_date|source_batch_id|ingestion_timestamp       |
+--------+-----------+----------+------+----------+---------------+--------------------------+
|B01_E001|1011       |Phuket    |125.0 |2026-05-01|batch_01       |2026-06-03 03:02:46.104331|
|B01_E002|1012       |Rayong    |150.0 |2026-05-01|batch_01       |2026-06-03 03:02:46.104331|
|B01_E003|1013       |Bangkok   |175.0 |2026-05-01|batch_01       |2026-06-03 03:02:46.104331|
|B01_E004|1014       |Chiang Mai|200.0 |2026-05-01|batch_01       |2026-06-03 03:02:46.104331|
|B02_E001|1021       |Rayong    |225.0 |2026-05-02|batch_02       |2026-06-03 03:03:06.607469|
|B02_E002|1022       |Bangkok   |250.0 |2026-05-02|batch_02       |2026-06-03 03:03:06.607469|
|B02_E003|1023       |Chiang Mai|275.0 |2026-05-02|batch_02       |2026-06-03 03:03:06.607469|


### Example 2: Inspect table metadata with `DESCRIBE DETAIL`

`DESCRIBE DETAIL` ช่วยดู metadata ระดับ table เช่น:

- table format
- table location
- number of files
- table size

สำหรับ Day 18 ให้โฟกัสที่ `numFiles` และ `sizeInBytes` เพื่อเริ่มเชื่อมโยง small files กับ physical layout ของ table

จาก lab นี้ เรา append ทั้งหมด 6 small batches และแต่ละ batch ใช้ `repartition(4)` ก่อน write

ผลลัพธ์ที่ได้ตรงกับ pattern ที่เราตั้งใจจำลองไว้คือ:

`6 batches × 4 DataFrame partitions per batch = 24 files`

ประเด็นสำคัญคือ table นี้ไม่ได้ใหญ่ในแง่ปริมาณข้อมูล แต่ข้อมูลถูกกระจายเป็นไฟล์เล็กหลายไฟล์เกินไป ทำให้ Spark ต้อง list files, ตรวจ metadata และจัดการ read tasks มากขึ้นตอน query

Note: `repartition(n)` กำหนดจำนวน partitions สำหรับการประมวลผลข้อมูลก่อน write แต่ไม่ได้ guarantee ว่าจะได้ `n` output files เสมอ โดยเฉพาะเมื่อข้อมูลในแต่ละ batch เล็กมาก

In [6]:
def show_table_detail(table_name: str):
    # Show selected Delta table metadata that is useful for table maintenance.
    return (
        spark.sql(f"DESCRIBE DETAIL {table_name}")
        .select("format", "numFiles", "sizeInBytes", "location")
    )

before_detail_df = show_table_detail(table_name)
before_detail_df.show(truncate=False)

StatementMeta(, 20797979-2d68-424b-8344-32f989291512, 8, Finished, Available, Finished, False)

+------+--------+-----------+--------------------------------------------------------------------------------------------------------------------------------------------------------+
|format|numFiles|sizeInBytes|location                                                                                                                                                |
+------+--------+-----------+--------------------------------------------------------------------------------------------------------------------------------------------------------+
|delta |24      |104070     |abfss://a5de4c5d-a2ed-4a31-923d-ff4fcd64006e@onelake.dfs.fabric.microsoft.com/8f868c6a-7fc0-4ade-a075-39cee1694ad9/Tables/day18_sales_events_small_files|
+------+--------+-----------+--------------------------------------------------------------------------------------------------------------------------------------------------------+



### Example 3: Inspect Delta history

หลาย small writes จะเห็นเป็นหลาย commit ใน Delta history

สำหรับ production debugging `DESCRIBE HISTORY` ช่วยดูว่า table ผ่าน operation อะไรมาบ้าง เช่น `WRITE`, `OPTIMIZE`, `VACUUM START`, `VACUUM END` และบาง operation อาจมี metrics ให้ใช้ตรวจสอบผลลัพธ์เพิ่มเติม

In [7]:
(
    spark.sql(f"DESCRIBE HISTORY {table_name}")
    .select("version", "timestamp", "operation", "operationMetrics")
    .orderBy(F.col("version").desc())
    .show(truncate=False)
)

StatementMeta(, 20797979-2d68-424b-8344-32f989291512, 9, Finished, Available, Finished, False)

+-------+-----------------------+----------------------+------------------------------------------------------------+
|version|timestamp              |operation             |operationMetrics                                            |
+-------+-----------------------+----------------------+------------------------------------------------------------+
|5      |2026-06-03 03:03:21.537|WRITE                 |{numFiles -> 4, numOutputRows -> 4, numOutputBytes -> 17345}|
|4      |2026-06-03 03:03:18.453|WRITE                 |{numFiles -> 4, numOutputRows -> 4, numOutputBytes -> 17345}|
|3      |2026-06-03 03:03:15.448|WRITE                 |{numFiles -> 4, numOutputRows -> 4, numOutputBytes -> 17345}|
|2      |2026-06-03 03:03:12.253|WRITE                 |{numFiles -> 4, numOutputRows -> 4, numOutputBytes -> 17345}|
|1      |2026-06-03 03:03:08.718|WRITE                 |{numFiles -> 4, numOutputRows -> 4, numOutputBytes -> 17345}|
|0      |2026-06-03 03:03:01.963|CREATE TABLE AS SELECT|

### Example 4: Run a normal read query before maintenance

Cell นี้ไม่ได้ benchmark performance จริง เพราะข้อมูล mock มีขนาดเล็กมาก แต่ช่วยให้เห็นว่า table ถูกใช้งานเหมือน Delta table ปกติ

ใน production การตัดสินใจทำ maintenance ควรดูจากหลักฐานจริง เช่น จำนวนไฟล์, ขนาด table, query ที่ใช้บ่อย และความถี่ในการเขียนข้อมูลเข้า table ไม่ใช่แค่ความรู้สึกว่า table ช้า

In [11]:
df_daily_region_summary = (
    spark.read.table(table_name)
    .groupBy("event_date", "region")
    .agg(
        F.count("event_id").alias("event_count"),
        F.sum("amount").alias("total_amount"),
    )
    .orderBy("event_date", "region")
)

df_daily_region_summary.show(truncate=False)

StatementMeta(, 20797979-2d68-424b-8344-32f989291512, 13, Finished, Available, Finished, False)

+----------+----------+-----------+------------+
|event_date|region    |event_count|total_amount|
+----------+----------+-----------+------------+
|2026-05-01|Bangkok   |1          |175.0       |
|2026-05-01|Chiang Mai|1          |200.0       |
|2026-05-01|Phuket    |1          |125.0       |
|2026-05-01|Rayong    |1          |150.0       |
|2026-05-02|Bangkok   |1          |250.0       |
|2026-05-02|Chiang Mai|1          |275.0       |
|2026-05-02|Phuket    |1          |300.0       |
|2026-05-02|Rayong    |1          |225.0       |
|2026-05-03|Bangkok   |1          |325.0       |
|2026-05-03|Chiang Mai|1          |350.0       |
|2026-05-03|Phuket    |1          |375.0       |
|2026-05-03|Rayong    |1          |400.0       |
|2026-05-04|Bangkok   |1          |500.0       |
|2026-05-04|Chiang Mai|1          |425.0       |
|2026-05-04|Phuket    |1          |450.0       |
|2026-05-04|Rayong    |1          |475.0       |
|2026-05-05|Bangkok   |1          |575.0       |
|2026-05-05|Chiang M

### Example 5: Explain the read plan briefly

`explain()` ไม่ได้บอกว่า small files เยอะเท่าไรโดยตรง แต่ช่วยให้เห็นว่า Spark จะอ่านจาก table และทำ aggregation อย่างไร

ใน Day 18 เราใช้ `explain()` เป็นตัวช่วยเสริม ไม่ใช่ตัววัด file health หลัก

In [12]:
df_daily_region_summary.explain(True)

StatementMeta(, 20797979-2d68-424b-8344-32f989291512, 14, Finished, Available, Finished, False)

== Parsed Logical Plan ==
'Sort ['event_date ASC NULLS FIRST, 'region ASC NULLS FIRST], true
+- Aggregate [event_date#5552, region#5550], [event_date#5552, region#5550, count(event_id#5548) AS event_count#5570L, sum(amount#5551) AS total_amount#5572]
   +- SubqueryAlias spark_catalog.lh_pyspark_challenge.day18_sales_events_small_files
      +- Relation spark_catalog.lh_pyspark_challenge.day18_sales_events_small_files[event_id#5548,customer_id#5549,region#5550,amount#5551,event_date#5552,source_batch_id#5553,ingestion_timestamp#5554] parquet

== Analyzed Logical Plan ==
event_date: date, region: string, event_count: bigint, total_amount: double
Sort [event_date#5552 ASC NULLS FIRST, region#5550 ASC NULLS FIRST], true
+- Aggregate [event_date#5552, region#5550], [event_date#5552, region#5550, count(event_id#5548) AS event_count#5570L, sum(amount#5551) AS total_amount#5572]
   +- SubqueryAlias spark_catalog.lh_pyspark_challenge.day18_sales_events_small_files
      +- Relation spark_catalo

### Example 6: Run `OPTIMIZE` to compact files

คำสั่ง `OPTIMIZE` จะพยายาม compact small files ใน Delta table ให้เหมาะกับการอ่านมากขึ้น

สำหรับ lab นี้ เรา run กับ table เฉพาะของ Day 18 เท่านั้น

> ห้ามทดลอง `OPTIMIZE` แบบสุ่มกับ production table โดยยังไม่รู้ว่า table นี้ถูกอ่าน/เขียนเมื่อไร มีงานอื่นใช้งานพร้อมกันไหม และผลกระทบที่คาดว่าจะเกิดคืออะไร

In [13]:
optimize_result_df = spark.sql(f"OPTIMIZE {table_name}")
optimize_result_df.show(truncate=False)

StatementMeta(, 20797979-2d68-424b-8344-32f989291512, 15, Finished, Available, Finished, False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|path                                                                                                                                                    |metrics                                                                                                                                                                                                                                                                                        |
+-----------------------------------------------------------------------------------------------------------------

### Example 7: Compare table detail after `OPTIMIZE`

หลัง `OPTIMIZE` ให้ตรวจ `numFiles`, `sizeInBytes` และ history อีกครั้ง

จาก lab นี้ ก่อน `OPTIMIZE` table มี `numFiles = 24` และ `sizeInBytes` ประมาณ 104 KB

หลัง `OPTIMIZE` ผลลัพธ์คือ `numFiles = 1` และ `sizeInBytes` ประมาณ 4.8 KB แปลว่า small files หลายไฟล์ถูก compact/rewrite ให้เหลือ active data file เพียงไฟล์เดียว

ประเด็นสำคัญคือ `OPTIMIZE` ไม่ได้เปลี่ยนจำนวน rows หรือ business data แต่ช่วยจัด physical file layout ของ Delta table ให้เหมาะกับการอ่านมากขึ้น

In [14]:
after_optimize_detail_df = show_table_detail(table_name)
after_optimize_detail_df.show(truncate=False)

(
    spark.sql(f"DESCRIBE HISTORY {table_name}")
    .select("version", "timestamp", "operation", "operationMetrics")
    .orderBy(F.col("version").desc())
    .show(truncate=False)
)

StatementMeta(, 20797979-2d68-424b-8344-32f989291512, 16, Finished, Available, Finished, False)

+------+--------+-----------+--------------------------------------------------------------------------------------------------------------------------------------------------------+
|format|numFiles|sizeInBytes|location                                                                                                                                                |
+------+--------+-----------+--------------------------------------------------------------------------------------------------------------------------------------------------------+
|delta |1       |4855       |abfss://a5de4c5d-a2ed-4a31-923d-ff4fcd64006e@onelake.dfs.fabric.microsoft.com/8f868c6a-7fc0-4ade-a075-39cee1694ad9/Tables/day18_sales_events_small_files|
+------+--------+-----------+--------------------------------------------------------------------------------------------------------------------------------------------------------+

+-------+-----------------------+----------------------+----------------------------

### Example 8: Preview `VACUUM` with `DRY RUN`

`VACUUM ... DRY RUN` ใช้ preview รายการไฟล์ที่คำสั่งจะลบ โดยยังไม่ลบจริง

ใน lab นี้เราใช้ retention default/safe window ที่ 168 hours หรือ 7 days ดังนั้น table ที่เพิ่งสร้างวันนี้จะยังไม่มี obsolete files ที่เกิน retention และถูกลบได้ทันที

In [15]:
vacuum_preview_df = spark.sql(f"VACUUM {table_name} RETAIN 168 HOURS DRY RUN")
vacuum_preview_df.show(truncate=False)

StatementMeta(, 20797979-2d68-424b-8344-32f989291512, 17, Finished, Available, Finished, False)

+----+
|path|
+----+
+----+



### Example 9: Actual `VACUUM` command as commented code

Actual `VACUUM` จะลบไฟล์จริงถ้าเข้าเงื่อนไข retention ดังนั้นใน notebook นี้จึง comment ไว้เป็น default

เมื่อต้องใช้จริง ให้ตอบคำถามก่อน:

- ต้องการ time travel ย้อนหลังได้กี่วัน?
- มี concurrent reader/writer หรือ long-running job ไหม?
- table นี้มี compliance/audit requirement ไหม?
- เคย run `DRY RUN` แล้วเข้าใจไฟล์ที่จะถูกลบหรือยัง?

In [ ]:
# Actual cleanup command. Keep commented for safety in a learning notebook.
# spark.sql(f"VACUUM {table_name} RETAIN 168 HOURS")

# More aggressive retention is intentionally NOT used in this notebook.
# Do not reduce retention or bypass safety checks on production tables.
# spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
# spark.sql(f"VACUUM {table_name} RETAIN 0 HOURS")

### Example 10: Create a simple maintenance decision note

สำหรับ production mindset ให้แยก action ให้ชัด:

- ป้องกัน small files ตั้งแต่ตอน write
- compact เมื่อ table เริ่ม fragmented
- vacuum เฉพาะเมื่อ stale files เกิน retention และเข้าใจผลต่อ recovery

In [18]:
maintenance_note_data = [
    (
        "small_files_detected",
        "Check DESCRIBE DETAIL, DESCRIBE HISTORY, and workload pattern",
        "Do not jump to VACUUM. First understand why files are small.",
    ),
    (
        "optimize_candidate",
        "Run OPTIMIZE on a lab or low-risk table first",
        "Useful when many small files hurt read efficiency or table layout.",
    ),
    (
        "vacuum_candidate",
        "Run VACUUM DRY RUN before actual VACUUM",
        "Use safe retention based on recovery needs and active table usage.",
    ),
    (
        "write_prevention",
        "Review write frequency, DataFrame partitioning, and file layout strategy",
        "Preventing too many small files can be cheaper than fixing them later.",
    ),
]

maintenance_note_schema = T.StructType([
    T.StructField("scenario", T.StringType(), False),
    T.StructField("recommended_check", T.StringType(), True),
    T.StructField("maintenance_note", T.StringType(), True),
])

df_maintenance_note = spark.createDataFrame(maintenance_note_data, maintenance_note_schema)
df_maintenance_note.show(truncate=False)

StatementMeta(, 20797979-2d68-424b-8344-32f989291512, 20, Finished, Available, Finished, False)

+--------------------+------------------------------------------------------------------------+----------------------------------------------------------------------+
|scenario            |recommended_check                                                       |maintenance_note                                                      |
+--------------------+------------------------------------------------------------------------+----------------------------------------------------------------------+
|small_files_detected|Check DESCRIBE DETAIL, DESCRIBE HISTORY, and workload pattern           |Do not jump to VACUUM. First understand why files are small.          |
|optimize_candidate  |Run OPTIMIZE on a lab or low-risk table first                           |Useful when many small files hurt read efficiency or table layout.    |
|vacuum_candidate    |Run VACUUM DRY RUN before actual VACUUM                                 |Use safe retention based on recovery needs and active table usage.    

## Quick recap

| Question | Answer |
|---|---|
| Small files problem คืออะไร? | Table มีไฟล์เล็กจำนวนมากจน metadata/file-operation overhead สูงขึ้น |
| `OPTIMIZE` ทำอะไร? | Compact/rewrite small files ให้เป็นไฟล์ที่เหมาะกับการอ่านมากขึ้น |
| `VACUUM` ทำอะไร? | ลบ stale/unreferenced files ที่เก่ากว่า retention threshold |
| `OPTIMIZE` ลบไฟล์เก่าทันทีไหม? | ไม่ การลบไฟล์เก่าเป็นหน้าที่ของ `VACUUM` |
| ทำไมต้องระวัง retention? | Retention สั้นเกินไปอาจลด time travel/recovery window และกระทบ active table usage |


## Exercises

### Exercise 1: Create another small-files table

สร้าง table ใหม่ชื่อ `day18_customer_events_maintenance_exercise` โดย append หลาย batch เล็ก ๆ

Requirements:

- ใช้ table name แยกจาก example หลัก
- append อย่างน้อย 4 batches
- แต่ละ batch มีอย่างน้อย 3 records
- ใช้ `DESCRIBE DETAIL` เพื่อตรวจ `numFiles`, `sizeInBytes`, `location`

Expected result:

- มี Delta table ใหม่สำหรับ exercise
- เห็น metadata ของ table หลัง small writes
- เข้าใจว่า `repartition(n)` ช่วยกำหนดจำนวน partitions สำหรับการประมวลผลข้อมูลก่อน write แต่ไม่ได้ guarantee ว่าจะได้ `n` output files เสมอ

In [24]:
exercise_table_name = "day18_customer_events_maintenance_exercise"

spark.sql(f"DROP TABLE IF EXISTS {exercise_table_name}")

exercise_schema = T.StructType([
    T.StructField("event_id", T.StringType(), False),
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("event_type", T.StringType(), True),
    T.StructField("event_date", T.StringType(), True),
    T.StructField("source_batch_id", T.StringType(), True),
])

def create_customer_event_batch(batch_number: int):
    rows = [
        (f"CX{batch_number:02d}_001", 2000 + batch_number, "login", f"2026-06-{batch_number:02d}", f"cx_batch_{batch_number:02d}"),
        (f"CX{batch_number:02d}_002", 2100 + batch_number, "profile_update", f"2026-06-{batch_number:02d}", f"cx_batch_{batch_number:02d}"),
        (f"CX{batch_number:02d}_003", 2200 + batch_number, "purchase", f"2026-06-{batch_number:02d}", f"cx_batch_{batch_number:02d}"),
    ]
    return spark.createDataFrame(rows, exercise_schema)

for batch_number in range(1, 5):
    (
        create_customer_event_batch(batch_number)
        .withColumn("event_date", F.to_date("event_date"))
        .withColumn("ingestion_timestamp", F.current_timestamp())
        .repartition(3)
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(exercise_table_name)
    )

spark.read.table(exercise_table_name).orderBy("source_batch_id", "event_id").show(truncate=False)
show_table_detail(exercise_table_name).show(truncate=False)

StatementMeta(, 20797979-2d68-424b-8344-32f989291512, 26, Finished, Available, Finished, False)

+--------+-----------+--------------+----------+---------------+--------------------------+
|event_id|customer_id|event_type    |event_date|source_batch_id|ingestion_timestamp       |
+--------+-----------+--------------+----------+---------------+--------------------------+
|CX01_001|2001       |login         |2026-06-01|cx_batch_01    |2026-06-03 04:08:33.367324|
|CX01_002|2101       |profile_update|2026-06-01|cx_batch_01    |2026-06-03 04:08:33.367324|
|CX01_003|2201       |purchase      |2026-06-01|cx_batch_01    |2026-06-03 04:08:33.367324|
|CX02_001|2002       |login         |2026-06-02|cx_batch_02    |2026-06-03 04:08:36.829389|
|CX02_002|2102       |profile_update|2026-06-02|cx_batch_02    |2026-06-03 04:08:36.829389|
|CX02_003|2202       |purchase      |2026-06-02|cx_batch_02    |2026-06-03 04:08:36.829389|
|CX03_001|2003       |login         |2026-06-03|cx_batch_03    |2026-06-03 04:08:38.774003|
|CX03_002|2103       |profile_update|2026-06-03|cx_batch_03    |2026-06-03 04:08

### Exercise 2: Optimize the exercise table and inspect history

Run `OPTIMIZE` กับ exercise table แล้วตรวจผลจาก `DESCRIBE DETAIL` และ `DESCRIBE HISTORY`

Requirements:

- run `OPTIMIZE`
- compare table detail หลัง optimize
- show history เฉพาะ `version`, `timestamp`, `operation`, `operationMetrics`

Expected result:

- เห็น `OPTIMIZE` เป็น operation ใน Delta history
- เริ่มอ่าน metrics ได้ว่าไฟล์ถูก rewrite/compact อย่างไร

In [25]:
exercise_optimize_result_df = spark.sql(f"OPTIMIZE {exercise_table_name}")
exercise_optimize_result_df.show(truncate=False)

show_table_detail(exercise_table_name).show(truncate=False)

(
    spark.sql(f"DESCRIBE HISTORY {exercise_table_name}")
    .select("version", "timestamp", "operation", "operationMetrics")
    .orderBy(F.col("version").desc())
    .show(truncate=False)
)

StatementMeta(, 20797979-2d68-424b-8344-32f989291512, 27, Finished, Available, Finished, False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|path                                                                                                                                                                |metrics                                                                                                                                                                                                                                                                             |
+---------------------------------------------------------------------------------------------------------------

## Common mistakes

1. **คิดว่า `VACUUM` ช่วยให้ query เร็วขึ้นโดยตรง**

   `VACUUM` เน้นลบ stale/unreferenced files เพื่อ reclaim storage ส่วน read performance มักเกี่ยวกับ file layout, compaction, partitioning, clustering และ query pattern มากกว่า

2. **Run `VACUUM` ด้วย retention ต่ำเกินไปบน production table**

   การลด retention โดยไม่เข้าใจ recovery requirement, time travel window และงานที่ยังอ่านหรือเขียน table อยู่ อาจทำให้ไฟล์ที่ยังจำเป็นต่อการ debug, rollback หรือ recovery ถูกลบเร็วเกินไป

3. **คิดว่า `OPTIMIZE` ลบไฟล์เก่าทันที**

   `OPTIMIZE` rewrite active files ให้ compact ขึ้น แต่ไฟล์เก่าที่ถูกแทนที่จะยังอยู่จนกว่าจะเข้าเงื่อนไข `VACUUM`

4. **Optimize ทุก table บ่อยเกินไป**

   `OPTIMIZE` มี cost เพราะต้อง rewrite data ควรใช้เมื่อมีเหตุผล เช่น small files เยอะ, table fragmented, หรือ read-heavy workload ได้ประโยชน์จริง

5. **ดูแค่ row count แต่ไม่ดู file count / history**

   Row count เท่าเดิมไม่ได้แปลว่า file layout ดี ควรดู `DESCRIBE DETAIL`, `DESCRIBE HISTORY` และ workload behavior ประกอบกัน

6. **Run maintenance command ผิด engine**

   คำสั่ง Delta maintenance เช่น `OPTIMIZE` และ `VACUUM` ควรรันใน Fabric Spark context เช่น Notebook หรือ Spark job ไม่ใช่คิดว่า Lakehouse SQL analytics endpoint จะรองรับเหมือนกันเสมอ

7. **แก้ small files ด้วย maintenance อย่างเดียว**

   Maintenance เป็นการแก้หลังเกิดปัญหา แต่ production ควรคิดเรื่อง write pattern, partitioning, repartition/coalesce (จะลงรายละเอียดใน Day 25), optimize write และ micro-batch frequency ตั้งแต่ design ด้วย


## Expected Output / Success Criteria

เมื่อจบ Day 18 ควรทำได้:

- อธิบายได้ว่า small files problem คืออะไรใน Lakehouse table
- สร้าง Delta table จากหลาย small append writes ได้
- ใช้ `DESCRIBE DETAIL` เพื่อตรวจ `numFiles`, `sizeInBytes` และ `location` ได้
- ใช้ `DESCRIBE HISTORY` เพื่อตรวจ table operations และ operation metrics ได้
- run `OPTIMIZE` กับ lab table และอธิบายว่าเป็น file compaction ได้
- ใช้ `VACUUM ... DRY RUN` เพื่อ preview cleanup โดยไม่ลบไฟล์จริงได้
- อธิบายความต่างระหว่าง `OPTIMIZE` และ `VACUUM` ได้
- อธิบายได้ว่า retention สั้นเกินไปอาจกระทบ time travel, recovery และ active table usage
- มี maintenance mindset ว่าต้องตรวจ table/file metadata ก่อนตัดสินใจ run maintenance command

## Final takeaway

> Table maintenance ไม่ใช่แค่การรันคำสั่งให้ครบ แต่คือการเข้าใจว่า table ถูกเขียนอย่างไร ถูกอ่านอย่างไร และต้องเก็บ recovery window ไว้นานแค่ไหน

จำ mindset หลักของ Day 18:

- Small files เกิดจาก write pattern ได้บ่อยใน incremental/micro-batch workloads
- `OPTIMIZE` ใช้ปรับ file layout และช่วยเรื่อง read efficiency
- `VACUUM` ใช้ลบ obsolete files เพื่อ reclaim storage หลังผ่าน retention
- `DRY RUN` ควรเป็น default habit ก่อน actual `VACUUM`
- อย่าแลก recovery/time travel safety กับการ cleanup ที่ aggressive เกินไป

## Optional cleanup

In [ ]:
# spark.sql("DROP TABLE IF EXISTS day18_sales_events_small_files")
# spark.sql("DROP TABLE IF EXISTS day18_customer_events_maintenance_exercise")